# state-dict-load — faded example 2: Remap by stripping the module. prefix

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `state-dict-load`. Running the beacon reports progress on the `Transfer: state_dict load` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Transfer: state_dict load` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`state-dict-load`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "state-dict-load"
DD_SUBTOPIC = "Transfer: state_dict load"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A DDP-saved checkpoint prefixes every key with `module.`. To load into a plain model you build a new dict that removes that prefix from each key before calling `load_state_dict`.

## Faded exercise 2

The model and prefixed checkpoint are given. Complete the dict comprehension that strips the leading `module.` from every key.

**Fill in:** build the remapped dict stripping the 'module.' prefix from each key

In [ ]:
import torch.nn as nn


t.manual_seed(0)
model = nn.Linear(4, 3)
raw_ckpt = {'module.weight': t.randn(3, 4), 'module.bias': t.randn(3)}
remapped = None  # TODO: build the remapped dict stripping the 'module.' prefix from each key
model.load_state_dict(remapped)

def _test():
    t.manual_seed(0)
    m = nn.Linear(4, 3)
    raw = {'module.weight': t.randn(3, 4), 'module.bias': t.randn(3)}
    rem = {k[len('module.'):]: v for k, v in raw.items()}
    assert sorted(remapped.keys()) == sorted(rem.keys()) == ['bias', 'weight'], f'keys: {sorted(remapped.keys())}'
    assert t.equal(model.weight.data, raw['module.weight']), 'weight must load after remap'
    assert t.equal(model.bias.data, raw['module.bias'])

try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn as nn


t.manual_seed(0)
model = nn.Linear(4, 3)
raw_ckpt = {'module.weight': t.randn(3, 4), 'module.bias': t.randn(3)}
remapped = {k[len('module.'):]: v for k, v in raw_ckpt.items()}
model.load_state_dict(remapped)
```
</details>